An embedding is a list of numbers that represents meaning
Instead of giving the computer the word "cat", we give it something like:
cat → [0.2, -0.4, 0.9, 0.1, -0.7, …]  (usually 300–4096 numbers)
The same model turns every word/sentence into this kind of list.
Why numbers?
Because computers are very good at comparing lists of numbers

Dimensions (also called dimensionality) in embeddings simply means: how many numbers are in the list (vector) that represents your text.
Most popular sizes right now (March 2026) is 384 → fast & cheap (good enough for many apps),
768–1024 → sweet spot for most production RAG (best quality/cost balance),
1536–3072 → high-end APIs when you need top accuracy,
4096 → bleeding-edge open models (but heavy on storage & speed)

We create the model only uses 3 dimensions (real models use hundreds or thousands).

man      1   0    0

women    1   9    0

King     8   0    1

queen    8   9    1

boys     1   0    9

girl     1   9    9

king – man + woman

Step by step (just normal math on each column):

king        [8, 0, 1]

minus man   [1, 0, 0]  → [8-1, 0-0, 1-0] = [7, 0, 1]

plus woman  [1, 9, 0]  → [7+1, 0+9, 1+0] = [8, 9, 1]

→ [8, 9, 1] is exactly the same as queen!


That action/process of quickly finding and pulling out the right page/paragraph/information on basis of user question from the models is Retriever
Retriever is the smart search helper that finds and gives you only the useful parts fast.




RAG stands for Retrieval-Augmented Generation. It is a framework that combines two powerful ideas:
Retrieval: Pulling relevant information from an external knowledge base (documents, databases, web, etc.).
Generation: Using a Large Language Model (LLM) like GPT, Llama, or Grok to create a natural response.
 RAG dynamically fetches fresh, relevant facts/information which is not in LLm.

RAG Architecture:-

Your have a source data: PDFs, Word files, web pages, company docs, Notion pages, database exports, etc in local.

some Tools that read and extract text from different formats (PDF → text, HTML → clean text, etc.) is documet loader

Break long documents into smaller "chunks" (e.g., 300–800 tokens each) so they fit in embedding models and LLM context windows.
Why? Whole books won't fit — small meaningful pieces work better.

Converts text chunks (and later the user query) into vectors (numbers that capture meaning) called embedding.

Stores all the chunk vectors + metadata + original text into vestor store. It Allows fast "similarity search" (find chunks closest in meaning to query).

The "search engine" part 
Takes user query for similarity search called retrivers

Insert the retrieved chunks into the LLM prompt is called argument or prompt enginnering.

The large language model (Grok, GPT-4o, Llama-3.1, Claude, etc.) that reads the augmented prompt and generates the final answer.And the clean, grounded answer sent back to the user (often with citations to chunks).




Indexing Phase (Offline – done first):
Raw Documents
→ Load & Extract Text
→ Chunk / Split
→ Embed each chunk → Vector
→ Store in Vector Database

Query Phase (Real-time – every user question):
User Question
→ Embed query → Vector
→ Retrieve top-k similar chunks from Vector DB
→ Build augmented prompt (question + retrieved chunks)
→ Send to LLM
→ LLM generates grounded response
→ Return to user

What is Chunking?
After loading raw documents (PDFs, web pages, etc.), you break long text into smaller pieces called chunks.
Why?

Embedding models have limits (e.g., 512–8192 tokens).
LLM context windows are limited (even 128k/1M is not enough for whole books).
Small, meaningful chunks → better similarity search.

Chunking is one of the most important decisions in RAG.
Wrong chunking = bad retrieval → bad answers (even with the best LLM).
Good chunking = much better relevance, fewer hallucinations, better context.

Chunking Stratgy:-

(1) Fixed chunk:- Split text every N characters or N tokens (e.g., every 500 characters or 300 tokens).
Add some overlap (e.g., 20–100 tokens) so context isn't lost at boundaries.

Pros:

Very fast and easy to implement.
Predictable chunk sizes.

Cons:

Often cuts sentences/ideas in half → "lost context".

No respect for meaning/structure → poor retrieval quality.

When to use:

Very uniform text (logs, CSV data).

Example:
Text: "The Taj Mahal was built by Shah Jahan. It is located in Agra."

Fixed 20 chars → bad cuts like "The Taj Mahal was buil" | "t by Shah Jahan. It is..."

(2)Recursive chunk:-Try to split on natural separators in this order: paragraphs > sentences > words > characters.
Stop when chunk fits inside max size (e.g., 500 tokens).
Add overlap.

How it works (LangChain style):

Split by "\n\n" (paragraphs)
If too big → split by "." (sentences)
If still big → split by " " (words)
If needed → characters

Pros:

Respects document structure better than fixed-size.
Rarely cuts mid-sentence.
Very good balance of speed & quality.

Cons:

Still rule-based → not truly semantic.

When to use:

80% of real projects start here (LangChain/LlamaIndex default).
Mixed text (books, articles, reports).

(3)Semantic chunking:-

Instead of splitting text by rules (like every 500 characters, or on periods/paragraphs like recursive does), semantic chunking splits based on meaning.
It asks: "Are these sentences talking about the same topic or has the idea changed?"
If the meaning changes a lot → new chunk starts.
If meanings are similar → keep them together.
This creates chunks that feel like natural paragraphs or sections — even if the original document has no clear breaks.

Original:
"The Taj Mahal is a white marble mausoleum. It was built by Shah Jahan. Construction took from 1632 to 1653. It is in Agra, India.
Meanwhile, the Red Fort is a historic fort. Shah Jahan also built it. It served as the main residence of Mughal emperors."

Semantic chunking might produce:
Chunk 1: "The Taj Mahal is a white marble mausoleum. It was built by Shah Jahan. Construction took from 1632 to 1653. It is in Agra, India."

Chunk 2: "Meanwhile, the Red Fort is a historic fort. Shah Jahan also built it. It served as the main residence of Mughal emperors."

→ Because sentences about Taj Mahal are very similar in embedding space, but "Meanwhile, the Red Fort..." has a clear topic shift.

(4)proposition-based chunking:-

Instead of splitting text by size, sentences, or similarity (like recursive or semantic),
proposition-based chunking uses an LLM to break the document into tiny, atomic facts called propositions.
A proposition = one single, clear, self-contained, verifiable statement/fact.

It should be atomic (can't be broken down further without losing meaning).
It should stand alone (no need for surrounding sentences to understand it).
Designed to be easy for embeddings + LLM to handle.

Example (real text → propositions):
Original paragraph:

"The Taj Mahal is a white marble mausoleum located in Agra, India. It was commissioned in 1632 by Mughal Emperor Shah Jahan in memory of his wife Mumtaz Mahal, and construction was completed in 1653."

After proposition chunking (LLM does this):

The Taj Mahal is a mausoleum made of white marble.

The Taj Mahal is located in Agra, India.

Shah Jahan commissioned the Taj Mahal in 1632.

Shah Jahan was a Mughal Emperor.

The Taj Mahal was built in memory of Mumtaz Mahal.

Mumtaz Mahal was Shah Jahan's wife.

Construction of the Taj Mahal was completed in 1653.

→ Each of these becomes its own chunk (or small group of 2–5 related propositions).

Vector Database;-

Your RAG system has thousands or millions of text chunks, each turned into a vector (a list of numbers representing meaning).
You need to answer questions like:
“Find the 5 chunks whose meaning is closest to my user query vector.”
Traditional databases (PostgreSQL, MongoDB, MySQL) are terrible at this because:

They are designed for exact matches (SQL WHERE clause).
Comparing millions of high-dimensional vectors with math (cosine/Euclidean) is extremely slow.

Vector Database = a specialized database built from the ground up to store vectors + do similarity search at lightning speed.

Exact search will compare every vector (O(n) time) → too slow for >10k vectors.

ANN (Approximate nearest neighbours) is smart approximation algorithms that give ~95–99% accurate results in milliseconds even for billions of vectors.

The Index (The Heart of Every Vector DB):- 

The index is a special data structure built during indexing.
Most popular in 2026:

HNSW (Hierarchical Navigable Small World) indexing  — graph-based, fastest & most accurate (used by Qdrant, Weaviate, Chroma, PGVector).

IVF (Inverted File) + PQ (Product Quantization) indexing— great for huge scale (Milvus, Pinecone).
Flat — exact search (only for small datasets).

When you insert vectors, the DB builds/updates this index automatically.

Basic Operation in vector Database:-

Upsert — add or update vector + metadata + original text.

Query — send query vector + optional filters → get top-k + scores + metadata.

Delete — by ID or filter.

Hybrid — vector similarity + keyword (BM25) in one call.

Important topic:-

(1)Offline: Chunk → Embed → Upsert into Vector DB (with metadata).

(2)Online: User query → Embed → (optional hybrid + filter) → Retrieve top-k → Send to LLM.

(3)Decision Tree (Simple 2026 Rule)

< 100k vectors + learning +small to meadium size ued → Chroma db (local)

Production, no infra team → Pinecone db (serverless)

Best open-source performance → Qdrant db

Need hybrid + graph features → Weaviate db

Billions of vectors → Milvus db

Already on Postgres → PGVector db

Pure research/speed → FAISS db



If you want managed & easy → Pinecone is still the most common "default" in many companies.

If you want open-source + scale → Milvus or Qdrant lead.

If you're already on Postgres → pgvector wins by default.

For your agentic AI projects  (learning phase):

Start with Chroma (local) or pgvector (if you use Supabase/Neon) → then test Qdrant or Pinecone free tier when going production.

Naive RAG (also called basic or classic RAG) is the simplest, most straightforward version — no fancy query rewriting, no reranking, no agents, no hybrid search. Just the core loop:
Load → Chunk → Embed → Store → Retrieve top-k → Stuff into prompt → Generate.

Two popular choices exist for naive Rag:

LangChain → more flexible chains & agent-friendly later

LlamaIndex → simpler, more focused on indexing & retrieval

Embeddings done by Hugging Face's all-MiniLM-L6-v2 (fast & free, 384 dim)

Vector DB used Chroma (easiest local DB, no API key needed)

LLM used ollama (or swap with local via Ollama/Groq later)

Lets start using langchain:-

Create a pdf folder and add a pdf after that install below command

uv add langchain langchain-community  langchain-huggingface
 langchain-text-splitters chromadb pypdf sentence-transformers 

In [18]:
import os
from langchain_community.document_loaders import PyPDFLoader          # or TextLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import OllamaLLM   # Note: this is the non-chat version
from pathlib import Path

# Option A – relative (recommended first)
pdf_path = Path("pdf") / "resttemplate.pdf"




loader = PyPDFLoader(str(pdf_path))
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    length_function=len,
)
chunks = text_splitter.split_documents(docs)

print(f"Loaded & chunked into {len(chunks)} pieces")

# -------------------------------
# 3. Embed & Store in Vector DB
# -------------------------------
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="basic_rag_collection",
    persist_directory="./chroma_db"   # saves to disk
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})   # top-4 chunks

# -------------------------------
# 4. Define Prompt Template
# -------------------------------
template = """You are a helpful assistant. Answer the question using ONLY the following context. 
If you don't know, say "I don't have enough information".

Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

# -------------------------------
# 5. LLM
# -------------------------------
llm = OllamaLLM(model="llama3.2")

# -------------------------------
# 6. Build the RAG Chain (LCEL style - clean & modern)
# -------------------------------
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# -------------------------------
# 7. Query Time!
# -------------------------------
question = "What is rest templet?"   # Change this
response = rag_chain.invoke(question)

print("\nQuestion:", question)
print("Answer:", response)

Loaded & chunked into 37 pieces


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8911.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Question: What is rest templet?
Answer: I don't have enough information.
